In [1]:
import pandas as pd
import numpy as np
import os
from glob import glob
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import warnings 
warnings.filterwarnings("ignore")

In [2]:
attendance_df = pd.read_csv(r"C:\Users\SHYAM\OneDrive\Desktop\ML Project\Merged Attendance.csv")
attendance_df.head()

,S.No,Enrollment No.,Student Name,Algo,ML,MCA,TOC,OS,Discrete Mathematics,Constitution,Lab - Algo,Lab - ML,Lab - OS,Total Classes,Percentage
0,NaN,NaN,NaN,16,8,5,17,16,8,5,10,2,2,89,100
1,1.0,21323001.0,Aashima,10,8,4,10,9,4,2,3,2,2,52,58.43%
2,2.0,21323002.0,Amit kumar,16,8,5,17,16,4,5,9,2,2,82,92.13%
3,3.0,21323004.0,Ansh Rathi,14,7,5,11,9,0,1,10,2,2,59,66.29%
4,4.0,21323005.0,Aryan raj,13,5,4,15,14,4,4,5,0,2,64,71.91%


In [3]:
lectures_df = pd.read_csv(r"C:\Users\SHYAM\OneDrive\Desktop\ML Project\lectures2.csv")
lectures_df

,Algo,ML,MCA,TOC,OS,Discrete Mathematics,Constitution,Lab - Algo,Lab - ML,Lab - OS,Total Classes
0,40,40,30,30,40,30,40,10,10,10,280


In [4]:
keep_columns = ['S.No', 'Enrollment No.', 'Student Name', 'Total Classes']  # Keep 'Total Classes'
attendance_df = attendance_df.drop(columns=['Percentage'], errors='ignore')  # Remove only 'Percentage'

selected_data = attendance_df[keep_columns]  # Select relevant columns
result_df = selected_data.copy()
result_df = result_df.drop(columns=['Total Classes'], errors='ignore')

result_df.to_csv(r"C:\Users\SHYAM\OneDrive\Desktop\ML Project\Result 2.csv", index=False)
df_result = pd.read_csv(r"C:\Users\SHYAM\OneDrive\Desktop\ML Project\Result 2.csv")
df_result.head()


,S.No,Enrollment No.,Student Name
0,NaN,NaN,NaN
1,1.0,21323001.0,Aashima
2,2.0,21323002.0,Amit kumar
3,3.0,21323004.0,Ansh Rathi
4,4.0,21323005.0,Aryan raj


In [5]:
attendance_folder = r"C:\Users\SHYAM\OneDrive\Desktop\ML Project\Attendances"
result_file = r"C:\Users\SHYAM\OneDrive\Desktop\ML Project\Result 2.csv"


# Read all CSV files from the folder and merge them
attendance_files = glob(os.path.join(attendance_folder, "*.csv"))

# Read and sum all attendance files
attendance_df = None
for file in attendance_files:
    temp_df = pd.read_csv(file)
    if attendance_df is None:
        attendance_df = temp_df
    else:
        # Sum attendance values for all subjects (except metadata columns)
        print("Attendance DF rows:", attendance_df.shape[0])
        print("Temp DF rows:", temp_df.shape[0])
        attendance_df.iloc[:, 3:] += temp_df.iloc[:, 3:]
        

# Read the result file and clean it
df_result = pd.read_csv(result_file)
df_result = df_result.drop(index=0).reset_index(drop=True)
df_result.head()

Attendance DF rows: 43
Temp DF rows: 43


,S.No,Enrollment No.,Student Name
0,1.0,21323001.0,Aashima
1,2.0,21323002.0,Amit kumar
2,3.0,21323004.0,Ansh Rathi
3,4.0,21323005.0,Aryan raj
4,5.0,21323006.0,Charvi Aggarwal


In [6]:
# Number of subjects
n_subjects = lectures_df.shape[1]

# Get subject columns
subject_columns = attendance_df.columns[3:3 + n_subjects]
bunk_columns = [f"{col} Bunks Available" for col in subject_columns]

# Ensure result file has necessary columns
for bunk_col in bunk_columns:
    if bunk_col not in df_result.columns:
        df_result[bunk_col] = 0

# Number of students
st = attendance_df.shape[0] - 1

j = 0;
# Compute and update bunk availability per student per subject
for k in range(0, st):
    for j, i in enumerate(range(3, 3 + n_subjects)):
        atten_sub = attendance_df.iloc[k + 1, i]  # Total attended lectures
        lec_sub = lectures_df.iloc[0, j]         # Total lectures for subject
        total_class = attendance_df.iloc[0, i]   # Total classes conducted
        
        # Calculate available bunks
        abscence = total_class - atten_sub
        # print("Abscence : ", abscence , end="    ")
        bunks_avail = (lec_sub * 0.25) - abscence
        # print("Total classes : ", lec_sub * 0.25 , end="    ")
        # Update the result DataFrame
        df_result.loc[k, bunk_columns[j]] = bunks_avail
    j += 1
    # print()

# Save the updated DataFrame back to the result CSV
df_result.to_csv(result_file, index=False)

print(f"Bunks availability successfully updated and saved in {result_file}")


Bunks availability successfully updated and saved in C:\Users\SHYAM\OneDrive\Desktop\ML Project\Result 2.csv


In [7]:
df_result = pd.read_csv(r"C:\Users\SHYAM\OneDrive\Desktop\ML Project\Result 2.csv")
df_result.head()

,S.No,Enrollment No.,Student Name,Algo Bunks Available,ML Bunks Available,MCA Bunks Available,TOC Bunks Available,OS Bunks Available,Discrete Mathematics Bunks Available,Constitution Bunks Available,Lab - Algo Bunks Available,Lab - ML Bunks Available,Lab - OS Bunks Available,Total Classes Bunks Available
0,1.0,21323001.0,Aashima,-2,10,5.5,-6.5,-4,-0.5,4,-11.5,2.5,2.5,-4
1,2.0,21323002.0,Amit kumar,10,10,7.5,7.5,10,-0.5,10,0.5,2.5,2.5,56
2,3.0,21323004.0,Ansh Rathi,6,8,7.5,-4.5,-4,-8.5,2,2.5,2.5,2.5,10
3,4.0,21323005.0,Aryan raj,4,4,5.5,3.5,6,-0.5,8,-7.5,-1.5,2.5,20
4,5.0,21323006.0,Charvi Aggarwal,4,8,5.5,-6.5,-4,-4.5,0,-3.5,2.5,2.5,0


In [8]:
df_result = pd.read_csv(r"C:\Users\SHYAM\OneDrive\Desktop\ML Project\Result 2.csv")
df_result.tail(6)

,S.No,Enrollment No.,Student Name,Algo Bunks Available,ML Bunks Available,MCA Bunks Available,TOC Bunks Available,OS Bunks Available,Discrete Mathematics Bunks Available,Constitution Bunks Available,Lab - Algo Bunks Available,Lab - ML Bunks Available,Lab - OS Bunks Available,Total Classes Bunks Available
36,37.0,21723009.0,Vaibhav Mishra,-16,-6,1.5,-26.5,-22,-8.5,0,-9.5,-1.5,-1.5,-90
37,38.0,21323040.0,Yashraj Lamba,-2,10,7.5,-8.5,-6,-0.5,2,-1.5,2.5,2.5,2
38,39.0,21323041.0,Yug Sharma,6,6,5.5,-8.5,-6,-4.5,6,-7.5,2.5,0.5,-2
39,40.0,NaN,Soumaya (LATERAL ENTRY),-20,-6,1.5,-26.5,-22,-8.5,0,-13.5,-1.5,-1.5,-98
40,41.0,NaN,Divyam (LATERAL ENTRY),0,6,1.5,5.5,8,-4.5,6,-9.5,2.5,2.5,14
41,42.0,21723001.0,Abhishek Saxena,8,6,5.5,-10.5,-8,-4.5,2,-1.5,2.5,2.5,-2
